In [1]:
import importlib
import json
import sys
import time
import warnings
from pathlib import Path

import numpy as np
import pandas as pd
import requests

warnings.filterwarnings("ignore", category=FutureWarning)

# Ensure project root is on sys.path so src/ imports work.
PROJECT_ROOT = str(Path(".").resolve())
if PROJECT_ROOT not in sys.path:
    sys.path.insert(0, PROJECT_ROOT)

# Reload so changes to injury_loader.py take effect without a kernel restart.
import src.data.injury_loader as _il_mod
importlib.reload(_il_mod)
from src.data.injury_loader import parse_injury_type
print("injury_loader loaded from:", _il_mod.__file__)

PROCESSED_DIR = Path("data/processed")
PROCESSED_DIR.mkdir(parents=True, exist_ok=True)

injury_loader loaded from: /Users/nateseluga/Pitcher-Injury-Risk/src/data/injury_loader.py


---
## 1 — Load Raw Data

In [2]:
# ── Statcast: load all parquet files under data/raw/statcast/ ────────────────
sc_files = sorted(Path("data/raw/statcast").glob("*.parquet"))
if not sc_files:
    raise FileNotFoundError("No Statcast parquet files found. Run notebook 01 first.")
sc_raw = pd.concat([pd.read_parquet(f) for f in sc_files], ignore_index=True)
sc_raw["game_date"] = pd.to_datetime(sc_raw["game_date"])
print(f"Statcast raw:  {sc_raw.shape[0]:>7,} rows  |  {sc_raw.shape[1]} columns  |  "
      f"{sc_raw['game_date'].dt.year.unique().tolist()} seasons")

# ── Injury database ─────────────────────────────────────────────────────────
inj_path = Path("data/raw/injuries/injury_database.parquet")
if not inj_path.exists():
    raise FileNotFoundError("No injury database found. Run notebook 02 first.")
inj_raw = pd.read_parquet(inj_path)
print(f"Injury DB raw: {len(inj_raw):>7,} stints  |  {inj_raw['player_id'].nunique()} pitchers")

# ── Player metadata ──────────────────────────────────────────────────────────
meta_path = Path("data/raw/player_metadata/pitchers.parquet")
if not meta_path.exists():
    raise FileNotFoundError("No player metadata found. Run notebook 01 first.")
meta_raw = pd.read_parquet(meta_path)
print(f"Metadata raw:  {len(meta_raw):>7,} pitchers  |  columns: {meta_raw.columns.tolist()}")

Statcast raw:   25,714 rows  |  80 columns  |  [2023] seasons
Injury DB raw:     244 stints  |  180 pitchers
Metadata raw:      400 pitchers  |  columns: ['player_id', 'key_fangraphs', 'key_bbref', 'name_last', 'name_first', 'birth_date', 'player_name']


---
## 2 — Statcast Cleaning

Steps applied in order:
1. Drop columns that are 100% null (no information content)
2. Deduplicate on the natural pitch key `(game_pk, at_bat_number, pitch_number)`
3. Standardize pitch type codes across eras (e.g. FT→SI, FA→FF from pre-2019 data)
4. Flag and remove rows with physically implausible values
5. Summarize null rates for modeling-relevant columns

In [3]:
# ── 2a: Drop always-null columns ─────────────────────────────────────────────
# spin_dir and sv_id are always null in pybaseball pulls and carry no signal.
always_null = [c for c in sc_raw.columns if sc_raw[c].isna().all()]
print(f"Always-null columns ({len(always_null)}): {always_null}")

# Also drop columns that are >99% null — they can't be useful features.
null_rate = sc_raw.isna().mean()
mostly_null = null_rate[null_rate > 0.99].index.tolist()
# Remove already-flagged duplicates.
mostly_null = [c for c in mostly_null if c not in always_null]
print(f"Mostly-null columns (>99%) ({len(mostly_null)}): {mostly_null}")

drop_cols = always_null + mostly_null
sc = sc_raw.drop(columns=drop_cols)
print(f"\nColumns after drop: {sc_raw.shape[1]} → {sc.shape[1]}")

Always-null columns (2): ['spin_dir', 'sv_id']
Mostly-null columns (>99%) (0): []

Columns after drop: 80 → 78


In [4]:
# ── 2b: Deduplication ────────────────────────────────────────────────────────
# A pitch is uniquely identified by (game_pk, at_bat_number, pitch_number).
# Duplicates arise when multiple parquet files overlap in date range.
PITCH_KEY = ["game_pk", "at_bat_number", "pitch_number"]
key_cols_present = [c for c in PITCH_KEY if c in sc.columns]

n_before = len(sc)
if len(key_cols_present) == len(PITCH_KEY):
    sc = sc.drop_duplicates(subset=PITCH_KEY)
    n_dropped = n_before - len(sc)
    print(f"Duplicate pitches removed: {n_dropped:,}  ({n_before:,} → {len(sc):,})")
else:
    missing = set(PITCH_KEY) - set(key_cols_present)
    print(f"WARNING: dedup skipped — missing key columns: {missing}")
    n_dropped = 0

sc = sc.reset_index(drop=True)
print(f"Rows after deduplication: {len(sc):,}")

Duplicate pitches removed: 0  (25,714 → 25,714)
Rows after deduplication: 25,714


In [5]:
# ── 2c: Pitch type standardization ───────────────────────────────────────────
# Statcast changed pitch type codes multiple times between 2015 and present.
# The most important remaps for multi-season consistency:
#   FT  → SI   (old two-seamer label, replaced by SI)
#   FA  → FF   (old generic fastball, replaced by FF)
#   FO  → FS   (forkball grouped with splitter)
#   CS  → CU   (slow curve grouped with curveball)
# Pitchouts (PO), intentional balls (IN), and automatic balls (AB) are not real
# pitches and are removed.

PITCH_TYPE_REMAP = {"FT": "SI", "FA": "FF", "FO": "FS", "CS": "CU"}
PITCH_TYPES_DROP = {"PO", "IN", "AB"}

print("Pitch type distribution BEFORE standardization:")
print(sc["pitch_type"].value_counts().to_string())

n_before = len(sc)
sc["pitch_type"] = sc["pitch_type"].replace(PITCH_TYPE_REMAP)
sc = sc[~sc["pitch_type"].isin(PITCH_TYPES_DROP)]

print(f"\nRows removed (pitchouts/intentionals): {n_before - len(sc):,}")
print("\nPitch type distribution AFTER standardization:")
print(sc["pitch_type"].value_counts().to_string())

Pitch type distribution BEFORE standardization:
pitch_type
FF    8413
SL    4175
SI    3826
CH    2757
FC    2062
CU    1679
ST    1574
FS     590
KC     452
SV      71
FA      23
FO      18
SC       4
CS       3

Rows removed (pitchouts/intentionals): 0

Pitch type distribution AFTER standardization:
pitch_type
FF    8436
SL    4175
SI    3826
CH    2757
FC    2062
CU    1682
ST    1574
FS     608
KC     452
SV      71
SC       4


In [6]:
# ── 2d: Implausible value check ───────────────────────────────────────────────
# Physical limits for pitch-tracking measurements.  Values outside these
# bounds are sensor/tracking artifacts, not real pitch data.
#
# release_speed:     40–105 mph  (eephus ~45, elite heater ~104)
# release_spin_rate: 1000–4000 rpm
# release_extension: 3.0–8.5 feet
# pfx_x, pfx_z:      ±3.0 feet (horizontal and vertical movement)

BOUNDS = {
    "release_speed":     (40,   105),
    "release_spin_rate": (1000, 4000),
    "release_extension": (3.0,  8.5),
    "pfx_x":             (-3.0, 3.0),
    "pfx_z":             (-3.0, 3.0),
}

outlier_mask = pd.Series(False, index=sc.index)
print("Implausible value counts (rows that violate bounds):")
for col, (lo, hi) in BOUNDS.items():
    if col not in sc.columns:
        print(f"  {col}: column not present, skipping")
        continue
    bad = sc[col].notna() & ((sc[col] < lo) | (sc[col] > hi))
    outlier_mask |= bad
    if bad.sum() > 0:
        print(f"  {col} [{lo}–{hi}]:  {bad.sum()} outliers")
        print(sc.loc[bad, ["pitcher", "player_name", "game_date", col]].head(5).to_string())
    else:
        print(f"  {col} [{lo}–{hi}]:  OK")

n_before = len(sc)
sc = sc[~outlier_mask].reset_index(drop=True)
print(f"\nRows removed (implausible values): {n_before - len(sc):,}")
print(f"Statcast rows remaining: {len(sc):,}")

Implausible value counts (rows that violate bounds):
  release_speed [40–105]:  OK
  release_spin_rate [1000–4000]:  101 outliers
      pitcher       player_name  game_date  release_spin_rate
823    669923     Kirby, George 2023-06-07                929
831    669923     Kirby, George 2023-06-07                707
834    669923     Kirby, George 2023-06-07                816
1622   676710  Crawford, Kutter 2023-06-07                939
1648   676710  Crawford, Kutter 2023-06-07                866
  release_extension [3.0–8.5]:  OK
  pfx_x [-3.0–3.0]:  OK
  pfx_z [-3.0–3.0]:  OK

Rows removed (implausible values): 101
Statcast rows remaining: 25,613


In [7]:
# ── 2e: Null rate summary for modeling-relevant columns ───────────────────────
# Many columns are null only for non-ball-in-play events (hc_x, launch_speed, etc.).
# That is expected and not a data quality issue — they'll be handled feature-by-feature
# during engineering.  The columns below are the ones we'll use directly in models;
# null rates above ~5% here warrant investigation.

MODELING_COLS = [
    "release_speed", "release_spin_rate", "release_extension",
    "pfx_x", "pfx_z", "plate_x", "plate_z",
    "release_pos_x", "release_pos_y", "release_pos_z",
    "effective_speed", "spin_axis",
    "pitch_type", "game_date", "pitcher",
]

present = [c for c in MODELING_COLS if c in sc.columns]
null_rates = sc[present].isna().mean().sort_values(ascending=False)
print("Null rates for modeling-relevant columns:")
for col, rate in null_rates.items():
    flag = "  ← review" if rate > 0.05 else ""
    print(f"  {col:<25}  {rate:.1%}{flag}")

Null rates for modeling-relevant columns:
  release_spin_rate          0.6%
  spin_axis                  0.6%
  plate_x                    0.3%
  plate_z                    0.3%
  pitch_type                 0.3%
  effective_speed            0.3%
  release_extension          0.3%
  release_speed              0.2%
  pfx_x                      0.2%
  pfx_z                      0.2%
  release_pos_x              0.2%
  release_pos_y              0.2%
  release_pos_z              0.2%
  game_date                  0.0%
  pitcher                    0.0%


---
## 3 — Injury Database Cleaning

Steps:
1. Audit stints classified as `other` — identify fixable gaps in the regex patterns
2. Re-apply `parse_injury_type` from the updated `injury_loader.py`
3. Review the days-lost distribution (median should be ~15–25 days)
4. Flag retroactive short-stints (<10 days) — these are real but unusual

In [8]:
# ── 3a: "Other" injury type audit (on the raw classifications) ────────────────
# Show all stints classified as 'other' and whether the description contains
# any recognisable body-part keyword that the old patterns missed.

print(f"Injury type distribution (raw, before re-classification):")
print(inj_raw["injury_type"].value_counts().to_string())
print()

others = inj_raw[inj_raw["injury_type"] == "other"].copy()
print(f"'Other' stints: {len(others)} of {len(inj_raw)} ({len(others)/len(inj_raw):.1%})")
print()

# Separate into: has a description with body-part text vs. truly undescribed.
has_desc = others[others["description"].str.contains(
    r"strain|contusion|inflam|fracture|tendin|sprain|tightness|soreness|pain|"
    r"fatigue|surgery|procedure|tear|rupture",
    case=False, na=False, regex=True
)]
no_desc = others[~others.index.isin(has_desc.index)]

print(f"  Has injury text in description: {len(has_desc)}  ← fixable via pattern update")
print(f"  No useful description:          {len(no_desc)}  ← irreducible (API omitted details)")
print()
print("Descriptions with injury text (showing first 20):")
for _, row in has_desc.head(20).iterrows():
    print(f"  {row.player_name}: {row.description}")

Injury type distribution (raw, before re-classification):
injury_type
other          47
shoulder       43
elbow          33
hip            20
back           18
forearm        16
oblique        15
hamstring      13
finger_hand    12
calf_ankle     12
knee            8
neck            5
illness         2

'Other' stints: 47 of 244 (19.3%)

  Has injury text in description: 27  ← fixable via pattern update
  No useful description:          20  ← irreducible (API omitted details)

Descriptions with injury text (showing first 20):
  Justin Verlander: New York Mets placed RHP Justin Verlander on the 15-day injured list retroactive to March 28, 2023. Low grade teres major strain.
  Jesse Chavez: Atlanta Braves placed RHP Jesse Chavez on the 15-day injured list. Left shin contusion.
  Daniel Bard: Colorado Rockies placed RHP Daniel Bard on the 15-day injured list. Right flexor strain.
  Max Scherzer: Texas Rangers placed RHP Max Scherzer on the 15-day injured list. Right teres major muscle str

In [9]:
# ── 3b: Re-apply updated injury classification ────────────────────────────────
# injury_loader.py was updated with expanded patterns:
#   - elbow:      added typos 'elobw', 'eblow' (real MLB API data errors)
#   - shoulder:   added teres major, deltoid, biceps tendinitis/inflammation, triceps
#   - forearm:    added 'flexor (muscle|tendon)? (strain|tear|...)' standalone pattern
#   - back:       added lat/latissimus, rhomboid, rib cage, pectoral
#   - calf_ankle: added shin, foot, toe, metatarsal
#   - illness:    added anxiety, cancer, facial fracture
#   - hip:        added SI joint / sacroiliac

inj = inj_raw.copy()
inj["injury_type"] = inj["description"].apply(parse_injury_type)

print("Injury type distribution BEFORE re-classification:")
before = inj_raw["injury_type"].value_counts()
print(before.to_string())
print()
print("Injury type distribution AFTER re-classification:")
after = inj["injury_type"].value_counts()
print(after.to_string())
print()

# Show the delta — which categories gained stints.
delta = after.subtract(before, fill_value=0).astype(int)
delta = delta[delta != 0].sort_values(ascending=False)
print("Net change per category (+ = gained, − = reduced):")
for cat, d in delta.items():
    sign = "+" if d > 0 else ""
    print(f"  {cat:<15}  {sign}{d}")

Injury type distribution BEFORE re-classification:
injury_type
other          47
shoulder       43
elbow          33
hip            20
back           18
forearm        16
oblique        15
hamstring      13
finger_hand    12
calf_ankle     12
knee            8
neck            5
illness         2

Injury type distribution AFTER re-classification:
injury_type
shoulder       53
elbow          35
back           26
hip            21
forearm        18
other          17
calf_ankle     16
oblique        15
hamstring      13
finger_hand    12
knee            8
illness         5
neck            5

Net change per category (+ = gained, − = reduced):
  shoulder         +10
  back             +8
  calf_ankle       +4
  illness          +3
  elbow            +2
  forearm          +2
  hip              +1
  other            -30


In [10]:
# ── 3c: Days-lost distribution ────────────────────────────────────────────────
# Expected shape: right-skewed with a floor around 10 days (minimum IL stint),
# a mode around 15-25 days, and a long tail for season-ending injuries.

paired = inj[inj["days_lost"].notna()]
season_ending = inj[inj["season_ending"] == True]

print(f"Total stints:          {len(inj):>5}")
print(f"Paired (days_lost):    {len(paired):>5}  ({len(paired)/len(inj):.1%})")
print(f"Season-ending (no act):{len(season_ending):>5}  ({len(season_ending)/len(inj):.1%})")
print()
print("Days-lost summary (paired stints only):")
print(paired["days_lost"].describe().to_frame().T.to_string())
print()

# Days-lost by injury type (median — easier to interpret than mean for skewed data).
print("Median days lost by injury type:")
medians = (
    paired.groupby("injury_type")["days_lost"]
    .agg(["median", "count"])
    .sort_values("median", ascending=False)
)
medians.columns = ["median_days", "n_stints"]
print(medians.to_string())

Total stints:            244
Paired (days_lost):      210  (86.1%)
Season-ending (no act):   34  (13.9%)

Days-lost summary (paired stints only):
           count       mean        std  min   25%   50%    75%    max
days_lost  210.0  37.566667  32.687996  4.0  15.0  24.0  51.75  152.0

Median days lost by injury type:
             median_days  n_stints
injury_type                       
neck                66.0         5
shoulder            36.0        43
forearm             34.0        14
illness             34.0         5
oblique             33.0        14
elbow               30.0        28
hip                 28.0        18
back                20.5        24
calf_ankle          16.5        14
hamstring           16.0        13
other               16.0        13
knee                15.5         8
finger_hand         15.0        11


In [11]:
# ── 3d: Flag short/retroactive stints (<10 days) ─────────────────────────────
# IL placements can be made retroactively — a pitcher placed on the IL on Sept 24
# but activated on Sept 28 shows 4 days lost even though the IL minimum is 10.
# These are legitimate transactions (end-of-season roster moves), not data errors.
# We flag them so feature engineering can treat them differently if needed.

SHORT_THRESH = 10  # days

inj["retroactive_short"] = (
    inj["days_lost"].notna()
    & (inj["days_lost"] < SHORT_THRESH)
)

short = inj[inj["retroactive_short"]]
print(f"Stints under {SHORT_THRESH} days: {len(short)} of {len(paired)} paired stints")
print()
print(short[["player_name", "transaction_date", "activation_date",
             "days_lost", "injury_type", "description"]].to_string())

Stints under 10 days: 11 of 210 paired stints

            player_name transaction_date activation_date  days_lost injury_type                                                                                                                           description
11          Daniel Bard       2023-09-27      2023-10-02        5.0     forearm                                              Colorado Rockies placed RHP Daniel Bard on the 15-day injured list. Right flexor strain.
13         Chris Martin       2023-09-28      2023-10-02        4.0     illness                                                   Boston Red Sox placed RHP Chris Martin on the 15-day injured list. Viral infection.
21      Scott Alexander       2023-09-23      2023-10-02        9.0   hamstring  San Francisco Giants placed LHP Scott Alexander on the 15-day injured list retroactive to September 22, 2023. Left hamstring strain.
23        Fernando Cruz       2023-09-01      2023-09-08        7.0       other                  

---
## 4 — Player Metadata Cleaning

The Chadwick Bureau register (used in notebook 01) returns only cross-reference IDs
(MLBAM, FanGraphs, BBRef). Birth dates are not included and came back as `NaT`.
Here we supplement from the MLB Stats API, which returns birth dates for all players
in one batch request.

In [12]:
# ── 4a: ID completeness ───────────────────────────────────────────────────────
meta = meta_raw.copy()

print(f"Total pitchers in metadata: {len(meta)}")
print()
print("Null rates per column:")
for col in meta.columns:
    rate = meta[col].isna().mean()
    flag = "  ← all null" if rate == 1.0 else ("  ← review" if rate > 0.1 else "")
    print(f"  {col:<20}  {rate:.1%}{flag}")

# ID completeness: how many pitchers have all three cross-reference IDs?
has_all_ids = meta[["player_id", "key_fangraphs", "key_bbref"]].notna().all(axis=1)
print(f"\nPitchers with all 3 IDs (MLBAM + FanGraphs + BBRef): {has_all_ids.sum()} / {len(meta)}")

Total pitchers in metadata: 400

Null rates per column:
  player_id             0.0%
  key_fangraphs         0.0%
  key_bbref             0.0%
  name_last             0.0%
  name_first            0.0%
  birth_date            100.0%  ← all null
  player_name           0.0%

Pitchers with all 3 IDs (MLBAM + FanGraphs + BBRef): 400 / 400


In [13]:
# ── 4b: Supplement birth dates from MLB Stats API ─────────────────────────────
# Endpoint: https://statsapi.mlb.com/api/v1/people?personIds=id1,id2,...
# Returns birthDate (YYYY-MM-DD) for each player.
# We batch requests at 200 IDs each to stay within URL length limits.

BATCH_SIZE = 200
BASE_URL = "https://statsapi.mlb.com/api/v1/people"

mlbam_ids = meta["player_id"].dropna().astype(int).tolist()
birth_map: dict[int, str] = {}

print(f"Fetching birth dates for {len(mlbam_ids)} pitchers in "
      f"{(len(mlbam_ids) + BATCH_SIZE - 1) // BATCH_SIZE} batches…")

for i in range(0, len(mlbam_ids), BATCH_SIZE):
    batch = mlbam_ids[i : i + BATCH_SIZE]
    params = {"personIds": ",".join(str(x) for x in batch),
              "fields": "people,id,birthDate"}
    try:
        resp = requests.get(BASE_URL, params=params, timeout=30)
        resp.raise_for_status()
        for person in resp.json().get("people", []):
            pid = person.get("id")
            bdate = person.get("birthDate")
            if pid and bdate:
                birth_map[int(pid)] = bdate
    except Exception as exc:
        print(f"  Batch {i//BATCH_SIZE + 1} failed: {exc}")
    time.sleep(0.5)

print(f"Birth dates retrieved: {len(birth_map)} / {len(mlbam_ids)}")

# Merge into metadata.
birth_series = meta["player_id"].map(birth_map)
meta["birth_date"] = pd.to_datetime(birth_series, errors="coerce")

still_null = meta["birth_date"].isna().sum()
print(f"Birth dates still null after supplement: {still_null}")
print(meta[["player_name", "birth_date"]].dropna().head(10).to_string())

Fetching birth dates for 400 pitchers in 2 batches…
Birth dates retrieved: 400 / 400
Birth dates still null after supplement: 0
        player_name birth_date
0     Rob Zastryzny 1992-03-26
1       Jakob Junis 1992-09-16
2      Héctor Neris 1989-06-14
3      Jesse Chavez 1983-08-21
4     Lucas Giolito 1994-07-14
5      Javier Assad 1997-07-30
6      Kyle Bradish 1996-09-12
7        Trevor May 1989-09-23
8  Jeremiah Estrada 1998-11-01
9     Steven Wilson 1994-08-24


---
## 5 — Cross-Source Join Validation

Checks that the three datasets connect cleanly on `player_id` (MLBAM).

**Note on TEST_MODE:** In TEST_MODE we have only 1 week of Statcast (June 1–7 2023)
but a full 2023 season of injuries. A pitcher who was injured in March will not appear
in the June Statcast slice. The low overlap here is *expected* — it will resolve when
the full season Statcast data is pulled.

In [14]:
# ── 5a: Statcast pitchers ↔ metadata ─────────────────────────────────────────
sc_pitchers = set(sc["pitcher"].dropna().astype(int).unique())
meta_pitchers = set(meta["player_id"].dropna().astype(int).unique())

in_sc_and_meta = sc_pitchers & meta_pitchers
in_sc_not_meta = sc_pitchers - meta_pitchers
in_meta_not_sc = meta_pitchers - sc_pitchers

print(f"Statcast unique pitchers:        {len(sc_pitchers):>5}")
print(f"Metadata unique pitchers:        {len(meta_pitchers):>5}")
print(f"In both (Statcast ∩ metadata):   {len(in_sc_and_meta):>5}  "
      f"({len(in_sc_and_meta)/len(sc_pitchers):.1%} of Statcast pitchers)")
print(f"In Statcast, not in metadata:    {len(in_sc_not_meta):>5}")
print(f"In metadata, not in Statcast:    {len(in_meta_not_sc):>5}  "
      f"(expected — metadata covers full season; Statcast is TEST_MODE slice)")

if in_sc_not_meta:
    # These pitchers appeared in a game but aren't in our metadata — flag them.
    print(f"\nStatcast pitchers missing from metadata (sample):")
    missing_names = sc[sc["pitcher"].isin(in_sc_not_meta)][["pitcher", "player_name"]]\
        .drop_duplicates().head(10)
    print(missing_names.to_string())

Statcast unique pitchers:          400
Metadata unique pitchers:          400
In both (Statcast ∩ metadata):     400  (100.0% of Statcast pitchers)
In Statcast, not in metadata:        0
In metadata, not in Statcast:        0  (expected — metadata covers full season; Statcast is TEST_MODE slice)


In [15]:
# ── 5b: Statcast pitchers ↔ injury database ───────────────────────────────────
inj_pitchers = set(inj["player_id"].dropna().astype(int).unique())

in_both   = sc_pitchers & inj_pitchers
inj_only  = inj_pitchers - sc_pitchers
sc_only   = sc_pitchers - inj_pitchers

print(f"Statcast unique pitchers:        {len(sc_pitchers):>5}")
print(f"Injury DB unique pitchers:       {len(inj_pitchers):>5}")
print(f"In both:                         {len(in_both):>5}")
print(f"Injured, not in Statcast slice:  {len(inj_only):>5}  "
      f"(normal — most injuries happened outside June 1-7)")
print(f"In Statcast, not injured:        {len(sc_only):>5}  "
      f"(pitchers who stayed healthy this week)")

if in_both:
    print(f"\nPitchers found in both Statcast and injury DB (sample):")
    both_names = inj[inj["player_id"].isin(in_both)][["player_id", "player_name",
                                                        "injury_type", "transaction_date"]]\
        .drop_duplicates("player_id").head(10)
    print(both_names.to_string())

Statcast unique pitchers:          400
Injury DB unique pitchers:         180
In both:                           180
Injured, not in Statcast slice:      0  (normal — most injuries happened outside June 1-7)
In Statcast, not injured:          220  (pitchers who stayed healthy this week)

Pitchers found in both Statcast and injury DB (sample):
    player_id       player_name  injury_type transaction_date
0      425794   Adam Wainwright          hip       2023-03-30
2      425844      Zack Greinke     shoulder       2023-07-05
4      434378  Justin Verlander     shoulder       2023-03-31
5      445276     Kenley Jansen        other       2023-09-13
6      445926      Jesse Chavez   calf_ankle       2023-06-15
7      446372      Corey Kluber     shoulder       2023-06-21
9      450203    Charlie Morton  finger_hand       2023-09-24
10     453268       Daniel Bard      forearm       2023-09-03
12     453286      Max Scherzer     shoulder       2023-09-13
13     455119      Chris Martin    

In [16]:
# ── 5c: Injury DB pitchers ↔ metadata ────────────────────────────────────────
inj_in_meta  = inj_pitchers & meta_pitchers
inj_not_meta = inj_pitchers - meta_pitchers

print(f"Injured pitchers in metadata:    {len(inj_in_meta):>5}  "
      f"({len(inj_in_meta)/len(inj_pitchers):.1%} of injured pitchers)")
print(f"Injured pitchers not in metadata:{len(inj_not_meta):>5}")

if inj_not_meta:
    # These pitchers had IL stints but no Statcast record — possibly position
    # players who pitched, two-way players, or players below the pitch threshold.
    print(f"\nInjured pitchers missing from metadata (sample):")
    missing_inj = inj[inj["player_id"].isin(inj_not_meta)][["player_id", "player_name",
                                                              "injury_type"]]\
        .drop_duplicates("player_id").head(10)
    print(missing_inj.to_string())

Injured pitchers in metadata:      180  (100.0% of injured pitchers)
Injured pitchers not in metadata:    0


In [17]:
# ── 5d: Coverage summary ──────────────────────────────────────────────────────
summary = {
    "Statcast pitches (cleaned)": len(sc),
    "Statcast unique pitchers": len(sc_pitchers),
    "Injury stints (cleaned)": len(inj),
    "Unique injured pitchers": len(inj_pitchers),
    "Pitcher metadata rows": len(meta),
    "Pitchers with birth_date": int(meta["birth_date"].notna().sum()),
    "Statcast ∩ metadata coverage": f"{len(in_sc_and_meta)/len(sc_pitchers):.1%}",
    "Injury DB ∩ metadata coverage": f"{len(inj_in_meta)/len(inj_pitchers):.1%}",
}

print("=" * 52)
print("DATA COVERAGE SUMMARY")
print("=" * 52)
for k, v in summary.items():
    print(f"  {k:<40}  {v}")
print("=" * 52)

DATA COVERAGE SUMMARY
  Statcast pitches (cleaned)                25613
  Statcast unique pitchers                  400
  Injury stints (cleaned)                   244
  Unique injured pitchers                   180
  Pitcher metadata rows                     400
  Pitchers with birth_date                  400
  Statcast ∩ metadata coverage              100.0%
  Injury DB ∩ metadata coverage             100.0%


---
## 6 — Save Cleaned Outputs

In [18]:
# ── Save all three cleaned datasets ──────────────────────────────────────────
sc_out   = PROCESSED_DIR / "statcast_clean.parquet"
inj_out  = PROCESSED_DIR / "injuries_clean.parquet"
meta_out = PROCESSED_DIR / "player_metadata_clean.parquet"

sc.to_parquet(sc_out, index=False)
inj.to_parquet(inj_out, index=False)
meta.to_parquet(meta_out, index=False)

print(f"Saved {sc_out}    — {len(sc):,} rows")
print(f"Saved {inj_out}   — {len(inj):,} rows")
print(f"Saved {meta_out}  — {len(meta):,} rows")

Saved data/processed/statcast_clean.parquet    — 25,613 rows
Saved data/processed/injuries_clean.parquet   — 244 rows
Saved data/processed/player_metadata_clean.parquet  — 400 rows


In [19]:
# ── Update provenance.json ────────────────────────────────────────────────────
from datetime import timezone, datetime

prov_path = Path("data/raw/provenance.json")
prov = json.loads(prov_path.read_text()) if prov_path.exists() else {}

prov["cleaning"] = {
    "run_at":               datetime.now(timezone.utc).isoformat(),
    "statcast_rows_raw":    len(sc_raw),
    "statcast_rows_clean":  len(sc),
    "statcast_cols_dropped": drop_cols,
    "injury_stints_raw":    len(inj_raw),
    "injury_stints_clean":  len(inj),
    "injury_other_before":  int(before.get("other", 0)),
    "injury_other_after":   int(after.get("other", 0)),
    "metadata_rows":        len(meta),
    "birth_dates_filled":   int(meta["birth_date"].notna().sum()),
}

prov_path.write_text(json.dumps(prov, indent=2, default=str))
print("Provenance updated:", prov_path)
print(json.dumps(prov["cleaning"], indent=2, default=str))

Provenance updated: data/raw/provenance.json
{
  "run_at": "2026-06-05T03:39:43.267059+00:00",
  "statcast_rows_raw": 25714,
  "statcast_rows_clean": 25613,
  "statcast_cols_dropped": [
    "spin_dir",
    "sv_id"
  ],
  "injury_stints_raw": 244,
  "injury_stints_clean": 244,
  "injury_other_before": 47,
  "injury_other_after": 17,
  "metadata_rows": 400,
  "birth_dates_filled": 400
}
